# 🌌 Résumé des alertes `sn_near_galaxy_candidate` — Fink/LSST

Pour chaque objet, ce notebook génère une **figure de synthèse** composée de :
- **Ligne 1** : courbe de lumière en flux PSF (nJy) + courbe de lumière en magnitude AB
- **Ligne 2** : cutouts Science / Template / Différence de la **dernière alerte** (source la plus récente)

Le titre de chaque figure indique le `diaObjectId`, le `diaSourceId`, le filtre et la date de l'alerte affichée.

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources` · `/api/v1/cutouts`

In [ ]:
# list des tags possibles
import requests, json

r = requests.get("https://api.lsst.fink-portal.org/api/v1/tags")
tags = r.json()
for tag, doc in tags.items():
    print(f"{tag:40s}  {doc}")

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_ALERTS   = 10                       # Nombre d'alertes à récupérer
STARTDATE  = None                     # "2026-01-01 00:00:00" ou None
STOPDATE   = None                     # "2026-03-01 00:00:00" ou None
BASE_URL   = "https://api.lsst.fink-portal.org"
TAG        = "extragalactic_lt20mag_candidate" # "sn_near_galaxy_candidate"
CUTOUT_CMAP   = 'hot'                 # Colormap pour les cutouts
CUTOUT_FORMAT = 'PNG'                 # Format API : 'PNG' ou 'FITS'
CUTOUT_KINDS  = ['Science', 'Template', 'Difference']
SAVE_FIGURES  = True                  # Sauvegarde PDF + PNG de chaque figure
OUTPUT_DIR    = "alert_summaries"     # Dossier de sortie
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
from io import BytesIO
from PIL import Image
from IPython.display import display as ipy_display
from IPython.display import HTML

MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')

def mjd_to_datetime(mjd_series):
    """Convertit une Series MJD en Series de Timestamps pandas (UTC)."""
    return MJD_EPOCH + pd.to_timedelta(pd.to_numeric(mjd_series, errors='coerce'), unit='D')

def mjd_to_datestr(mjd):
    """Convertit un scalaire MJD en chaîne 'YYYY-MM-DD'."""
    try:
        return (MJD_EPOCH + pd.to_timedelta(float(mjd), unit='D')).strftime('%Y-%m-%d')
    except Exception:
        return str(mjd)

if SAVE_FIGURES:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Imports OK ✓")

## 3. Style matplotlib

In [ ]:
mpl.rcParams.update({
    'font.size'         : 11,
    'axes.titlesize'    : 16,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 16,
    'xtick.labelsize'   : 16,
    'ytick.labelsize'   : 16,
    'legend.fontsize'   : 16,
    'figure.titlesize'  : 13,
    'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE',
    'g': '#0077BB',
    'r': '#33AA77',
    'i': '#DDAA33',
    'z': '#BB5500',
    'y': '#AA0000',
}
ZP_AB = 2.5 * np.log10(3631e9)   # ≈ 31.4 pour F en nJy

print("Style OK ✓")

## 4. Fonctions API

In [ ]:
def fetch_tag_alerts(tag, n, startdate=None, stopdate=None):
    """Récupère les N dernières alertes pour un tag donné."""
    payload = {"tag": tag, "n": n, "output-format": "json"}
    if startdate:
        payload["startdate"] = startdate
    if stopdate:
        payload["stopdate"] = stopdate
    resp = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        raise ValueError(f"Aucune alerte pour le tag '{tag}'.")
    return pd.DataFrame(data)


def fetch_lightcurve(dia_object_id):
    """Télécharge la courbe de lumière complète d'un diaObjectId."""
    payload = {
        "diaObjectId"  : str(dia_object_id),
        "columns"      : "r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band,r:diaSourceId",
        "output-format": "json",
    }
    resp = requests.post(f"{BASE_URL}/api/v1/sources", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df.columns = [c.replace('r:', '') for c in df.columns]
    for col in ['midpointMjdTai', 'psfFlux', 'psfFluxErr']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def fetch_cutout_image(dia_source_id, kind, output_format=CUTOUT_FORMAT):
    """Récupère un cutout PIL.Image pour un diaSourceId et un type donnés."""
    params = {
        "diaSourceId"  : str(dia_source_id),
        "kind"         : kind,
        "output-format": output_format,
    }
    resp = requests.get(f"{BASE_URL}/api/v1/cutouts", params=params, timeout=60)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content))


print("Fonctions API OK ✓")

## 5. Fonctions de tracé

In [ ]:
def flux_to_mag_ab(flux_njy, flux_err_njy):
    """Flux (nJy) → magnitude AB + erreur propagée. NaN si flux ≤ 0."""
    flux   = np.asarray(flux_njy,     dtype=float)
    flux_e = np.asarray(flux_err_njy, dtype=float)
    valid   = flux > 0
    mag     = np.full(flux.shape, np.nan)
    mag_err = np.full(flux.shape, np.nan)
    mag[valid]     = -2.5 * np.log10(flux[valid]) + ZP_AB
    mag_err[valid] = (2.5 / np.log(10)) * np.abs(flux_e[valid] / flux[valid])
    return mag, mag_err


def _prepare_lc(df):
    """Nettoyage commun : conversion types, drop NaN, ajout colonne date."""
    df = df.copy()
    for col in ['midpointMjdTai', 'psfFlux', 'psfFluxErr']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['midpointMjdTai', 'psfFlux', 'psfFluxErr'])
    df['date'] = mjd_to_datetime(df['midpointMjdTai'])
    return df


def _apply_date_axis(ax):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.grid(True, alpha=0.2, linewidth=0.5)


def plot_flux(ax, df, obj_id):
    """Courbe de lumière en flux PSF (nJy) — sans titre ni xlabel."""
    required = {'midpointMjdTai', 'psfFlux', 'psfFluxErr', 'band'}
    if not required.issubset(df.columns):
        ax.text(0.5, 0.5, "Données manquantes", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    df = _prepare_lc(df)
    if df.empty:
        ax.text(0.5, 0.5, "Pas de données valides", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    bands = sorted(df['band'].dropna().unique())
    for band in bands:
        sub   = df[df['band'] == band].sort_values('date')
        color = FILTER_COLORS.get(band, '#888888')
        ax.errorbar(sub['date'], sub['psfFlux'], yerr=sub['psfFluxErr'],
                    fmt='o', color=color, label=f"${band}$",
                    markersize=4, capsize=3, capthick=0.8,
                    elinewidth=0.8, linewidth=0.8, alpha=0.85)
    ax.axhline(0, color='gray', lw=0.6, ls='--', alpha=0.5)
    # Pas de titre (redondant avec le suptitle) ; pas de xlabel (redondant)
    ax.set_ylabel("Flux PSF (nJy)")
    ax.legend(ncol=len(bands), loc='upper left',
              handlelength=1, handletextpad=0.4, columnspacing=0.6)
    _apply_date_axis(ax)


def plot_mag(ax, df, obj_id):
    """Courbe de lumière en magnitude AB — sans titre ni xlabel."""
    required = {'midpointMjdTai', 'psfFlux', 'psfFluxErr', 'band'}
    if not required.issubset(df.columns):
        ax.text(0.5, 0.5, "Données manquantes", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    df = _prepare_lc(df)
    if df.empty:
        ax.text(0.5, 0.5, "Pas de données valides", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    mag, mag_err = flux_to_mag_ab(df['psfFlux'].values, df['psfFluxErr'].values)
    df['mag']     = mag
    df['mag_err'] = mag_err
    bands = sorted(df['band'].dropna().unique())
    for band in bands:
        sub   = df[df['band'] == band].sort_values('date')
        color = FILTER_COLORS.get(band, '#888888')
        det   = sub.dropna(subset=['mag'])
        if not det.empty:
            ax.errorbar(det['date'], det['mag'], yerr=det['mag_err'],
                        fmt='o', color=color, label=f"${band}$",
                        markersize=4, capsize=3, capthick=0.8,
                        elinewidth=0.8, linewidth=0.8, alpha=0.85)
        nondet = sub[sub['mag'].isna()]
        if not nondet.empty:
            sigma = nondet['psfFluxErr'].values
            v     = sigma > 0
            lim   = np.full(sigma.shape, np.nan)
            lim[v] = -2.5 * np.log10(3 * sigma[v]) + ZP_AB
            ax.scatter(nondet['date'], lim,
                       marker='v', color=color, s=25, alpha=0.4, zorder=3)
    ax.invert_yaxis()
    # Pas de titre ; pas de xlabel
    ax.set_ylabel("Magnitude AB")
    ax.legend(ncol=len(bands), loc='upper left',
              handlelength=1, handletextpad=0.4, columnspacing=0.6)
    _apply_date_axis(ax)


def plot_cutouts(axes_row, dia_source_id, kinds=CUTOUT_KINDS, cmap=CUTOUT_CMAP):
    """
    Remplit une rangée d'axes avec les cutouts Science / Template / Difference.
    aspect='auto' permet aux axes de remplir tout l'espace alloué par GridSpec
    sans laisser de marges blanches dues au ratio carré de imshow.
    Retourne la liste des (ax, im, array) réussis pour la normalisation commune.
    """
    arrays = []
    ax_arr_pairs = []
    for ax, kind in zip(axes_row, kinds):
        try:
            img = fetch_cutout_image(dia_source_id, kind)
            arr = np.array(img).astype(float)
            if arr.ndim == 3:
                arr = arr.mean(axis=2)
            arrays.append(arr)
            im = ax.imshow(arr, cmap=cmap, origin='upper', aspect='auto')
            ax.axis('off')
            ax_arr_pairs.append((ax, im, arr))
        except Exception as e:
            ax.axis('off')
            ax.text(0.5, 0.5, f"{kind}\nerreur\n{e}",
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=8, color='red')
    # Échelle de couleur commune
    if arrays:
        vmin = min(a.min() for a in arrays)
        vmax = max(a.max() for a in arrays)
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        for ax, im, arr in ax_arr_pairs:
            im.set_norm(norm)
    return ax_arr_pairs


print("Fonctions de tracé OK ✓")


## 6. Récupération des données

In [ ]:
from tqdm.notebook import tqdm

# ── Alertes ──
print(f"Récupération de {N_ALERTS} alertes '{TAG}'...")
df_tags = fetch_tag_alerts(TAG, N_ALERTS, STARTDATE, STOPDATE)
print(f"✓ {len(df_tags)} alertes reçues")

id_col = [c for c in df_tags.columns if 'diaObjectId' in c][0]
object_ids = df_tags[id_col].astype(str).unique().tolist()
print(f"Colonne ID : '{id_col}' — {len(object_ids)} objet(s) uniques\n")

# ── Courbes de lumière ──
lightcurves = {}
for oid in tqdm(object_ids, desc='Récupération coordonnées', unit='obj'):
for oid in object_ids:
    try:
        lc = fetch_lightcurve(oid)
        if lc.empty:
            print(f"  ⚠️  {oid} — aucune donnée")
        else:
            lightcurves[oid] = lc
            bands = sorted(lc['band'].dropna().unique()) if 'band' in lc.columns else []
            print(f"  ✓  {oid} — {len(lc)} sources, filtres: {bands}")
    except Exception as e:
        print(f"  ✗  {oid} — erreur: {e}")

print(f"\n{len(lightcurves)} objet(s) chargé(s) avec succès.")

## 7. Figures de synthèse

Pour chaque objet : courbes de lumière (flux + magnitude) sur la première ligne,  
cutouts de la **source la plus récente** sur la seconde ligne.

In [ ]:
for oid, lc in lightcurves.items():

    # ── Identifier la source la plus récente ──────────────────────────────────
    lc_sorted = lc.dropna(subset=['midpointMjdTai']).sort_values(
        'midpointMjdTai', ascending=False)

    if lc_sorted.empty or 'diaSourceId' not in lc_sorted.columns:
        print(f"⚠️  {oid} — impossible d'identifier la dernière source, objet ignoré.")
        continue

    last_row     = lc_sorted.iloc[0]
    last_src_id  = last_row['diaSourceId']
    last_band    = last_row.get('band', 'N/A')
    last_mjd     = last_row.get('midpointMjdTai', None)
    last_datestr = mjd_to_datestr(last_mjd) if last_mjd is not None else 'N/A'

    # ── Titre principal ───────────────────────────────────────────────────────
    main_title = (
        f"diaObjectId : {oid}   |   "
        f"Dernière alerte — diaSourceId : {last_src_id}   "
        f"filtre : {last_band}   date : {last_datestr}"
    )

    # ── Mise en page ──────────────────────────────────────────────────────────
    from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

    # constrained_layout gère automatiquement les marges et espacements,
    # ce qui évite tout chevauchement entre tick labels, titres et axes.
    # On fixe uniquement le ratio de hauteur LC / cutouts.
    FIG_W    = 16.0
    CUT_SIZE = FIG_W * 0.90 * 0.94 / 3   # côté d'un cutout carré estimé
    LC_H     = CUT_SIZE * 0.90            # LC un peu moins hautes que les cutouts

    fig = plt.figure(figsize=(FIG_W, LC_H + CUT_SIZE+1),
                     layout='constrained')
    fig.suptitle(main_title, fontsize=12, fontweight='bold')

    # Grille principale 2 lignes — constrained_layout s'occupe des espacements
    gs_main = GridSpec(2, 1, figure=fig,
                       height_ratios=[LC_H, CUT_SIZE])

    # Ligne 0 — deux courbes de lumière, même largeur
    gs_lc = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_main[0], wspace=0.07)
    ax_flux = fig.add_subplot(gs_lc[0])
    ax_mag  = fig.add_subplot(gs_lc[1])

    # Ligne 1 — trois cutouts + slot colorbar
    gs_bottom = GridSpecFromSubplotSpec(
        1, 2, subplot_spec=gs_main[1],
        width_ratios=[0.94, 0.06], wspace=0.02)
    gs_cut = GridSpecFromSubplotSpec(
        1, 3, subplot_spec=gs_bottom[0], wspace=0.03)
    ax_sci  = fig.add_subplot(gs_cut[0])
    ax_tpl  = fig.add_subplot(gs_cut[1])
    ax_dif  = fig.add_subplot(gs_cut[2])
    ax_cbar = fig.add_subplot(gs_bottom[1])
    ax_cbar.set_visible(False)

    # ── Tracé des courbes de lumière ──────────────────────────────────────────
    plot_flux(ax_flux, lc, oid)
    plot_mag(ax_mag,  lc, oid)

    # ── Tracé des cutouts ─────────────────────────────────────────────────────
    cutout_axes = [ax_sci, ax_tpl, ax_dif]
    ax_arr_pairs = plot_cutouts(
        cutout_axes, last_src_id, kinds=CUTOUT_KINDS, cmap=CUTOUT_CMAP)

    for ax, kind in zip(cutout_axes, CUTOUT_KINDS):
        ax.set_title(f"{kind}\nfiltre : {last_band}   {last_datestr}", fontsize=10)

    # ── Colorbar calée sur ax_dif après calcul des positions ──────────────────
    if ax_arr_pairs:
        norm_cut = plt.Normalize(
            vmin=min(a.min() for _, _, a in ax_arr_pairs),
            vmax=max(a.max() for _, _, a in ax_arr_pairs))

        fig.canvas.draw()
        pos_dif       = ax_dif.get_position()
        pos_cbar_slot = ax_cbar.get_position()

        cbar_ax = fig.add_axes([
            pos_cbar_slot.x0 + pos_cbar_slot.width * 0.10,
            pos_dif.y0,
            pos_cbar_slot.width * 0.35,
            pos_dif.height,
        ])
        sm = plt.cm.ScalarMappable(cmap=CUTOUT_CMAP, norm=norm_cut)
        sm.set_array([])
        fig.colorbar(sm, cax=cbar_ax, label='ADU')

    # ── Sauvegarde ────────────────────────────────────────────────────────────
    if SAVE_FIGURES:
        stem = f"{OUTPUT_DIR}/summary_{oid}_{last_src_id}"
        fig.savefig(f"{stem}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{stem}.png", bbox_inches='tight', dpi=150)
        #print(f"  ✓ Sauvegardé : {stem}.pdf / .png")

    plt.show()

    # ── Lien cliquable vers le portail Fink/LSST ──────────────────────────────
    fink_url = f"https://lsst.fink-portal.org/{oid}"
    ipy_display(HTML(
        f'<div style="font-size:12px; margin:2px 0 14px 4px;">'
        f'🔗 <b>Portail Fink/LSST</b> — '
        f'diaObjectId <code>{oid}</code> : '
        f'<a href="{fink_url}" target="_blank">{fink_url}</a>'
        f'</div>'
    ))
    print()

print("\n✅ Toutes les figures de synthèse ont été générées.")

## 8. (Optionnel) Tableau récapitulatif

In [ ]:
from IPython.display import HTML

rows = []
for oid, lc in lightcurves.items():
    lc_s = lc.dropna(subset=['midpointMjdTai']).sort_values(
        'midpointMjdTai', ascending=False)
    last = lc_s.iloc[0] if not lc_s.empty else {}
    rows.append({
        'diaObjectId'      : oid,
        'n_sources'        : len(lc),
        'filtres'          : ', '.join(sorted(lc['band'].dropna().unique())) if 'band' in lc.columns else '—',
        'MJD_min'          : lc['midpointMjdTai'].min(),
        'MJD_max'          : lc['midpointMjdTai'].max(),
        'flux_max (nJy)'   : lc['psfFlux'].max(),
        'last_diaSourceId' : last.get('diaSourceId', '—'),
        'last_band'        : last.get('band', '—'),
        'last_date'        : mjd_to_datestr(last.get('midpointMjdTai')) if 'midpointMjdTai' in last else '—',
    })

df_summary = pd.DataFrame(rows)

# Colonne lien cliquable (HTML — pas besoin de jinja2)
df_summary['Portail Fink'] = df_summary['diaObjectId'].apply(
    lambda oid: f'<a href="https://lsst.fink-portal.org/{oid}" target="_blank">'
                f'lsst.fink-portal.org/{oid}</a>'
)

# Rendu HTML avec liens cliquables
html = df_summary.to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse: collapse; font-size: 12px; width: 100%; }
  .fink-table th { background: #f0f0f0; padding: 6px 10px;
                   border-bottom: 2px solid #ccc; text-align: left; }
  .fink-table td { padding: 4px 10px; border-bottom: 1px solid #eee;
                   text-align: left; white-space: nowrap; }
  .fink-table tr:hover td { background: #fafafa; }
</style>
"""
ipy_display(HTML(style + html))
